In [ ]:
# Example 1: Basic YAML Parsing with Type Safety
from strictyaml import load, Str, Int, Float, Bool, Map, Seq, YAML

# Basic schema-driven parsing
basic_schema = Map({
    "name": Str(),
    "age": Int(),
    "height": Float(),
    "active": Bool(),
    "tags": Seq(Str())
})

basic_yaml = """
name: John Doe
age: 30
height: 5.9
active: true
tags:
  - developer
  - python
  - yaml
"""

# Parse with schema validation
try:
    parsed = load(basic_yaml, basic_schema)
    
    print("Basic StrictYAML Parsing:")
    print(f"Name: {parsed['name'].data} (type: {type(parsed['name'].data)})")
    print(f"Age: {parsed['age'].data} (type: {type(parsed['age'].data)})")
    print(f"Height: {parsed['height'].data} (type: {type(parsed['height'].data)})")
    print(f"Active: {parsed['active'].data} (type: {type(parsed['active'].data)})")
    print(f"Tags: {[tag.data for tag in parsed['tags']]} (length: {len(parsed['tags'])})")
    
    # Demonstrate type safety
    print(f"\nType Safety Verification:")
    print(f"Age is integer: {isinstance(parsed['age'].data, int)}")
    print(f"Height is float: {isinstance(parsed['height'].data, float)}")
    print(f"Active is boolean: {isinstance(parsed['active'].data, bool)}")
    
    # Show YAML object properties
    print(f"\nYAML Object Properties:")
    print(f"YAML type: {type(parsed)}")
    print(f"Has line numbers: {hasattr(parsed, 'start_mark')}")
    
except Exception as e:
    print(f"Parsing error: {e}")
    raise AssertionError(
        f"strictyaml example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Basic YAML parsing with type safety completed")

## Example 2: Advanced Schema Validation and Error Handling

In [ ]:
# Example 2: Advanced Schema Validation and Error Handling
from strictyaml import Optional, Email, Url, Regex, CommaSeparated, YAMLValidationError

# Advanced schema with optional fields and custom validators
advanced_schema = Map({
    "user": Map({
        "name": Str(),
        "email": Email(),
        "website": Url(),
        Optional("phone"): Regex(r"^\+?[\d\s\-\(\)]+$"),
        Optional("bio"): Str()
    }),
    "settings": Map({
        "theme": Str(),
        "notifications": Bool(),
        "languages": CommaSeparated(Str()),
        Optional("timeout"): Int()
    }),
    "permissions": Seq(Str())
})

# Valid YAML document
valid_yaml = """
user:
  name: Alice Johnson
  email: alice@example.com
  website: https://alice.dev
  phone: +1-555-0123
  bio: Software engineer and YAML enthusiast
settings:
  theme: dark
  notifications: true
  languages: python,yaml,json
  timeout: 30
permissions:
  - read
  - write
  - admin
"""

# Invalid YAML documents for error testing
invalid_yamls = {
    "invalid_email": """
user:
  name: Bob
  email: not-an-email
  website: https://bob.dev
settings:
  theme: light
  notifications: false
  languages: python
permissions:
  - read
""",
    
    "invalid_url": """
user:
  name: Charlie
  email: charlie@example.com
  website: not-a-url
settings:
  theme: blue
  notifications: true
  languages: java,python
permissions:
  - read
""",
    
    "invalid_type": """
user:
  name: David
  email: david@example.com
  website: https://david.dev
settings:
  theme: dark
  notifications: "not-a-boolean"
  languages: python,go
permissions:
  - read
"""
}

print("Advanced Schema Validation:")
print("=" * 40)

# Test valid YAML
try:
    parsed = load(valid_yaml, advanced_schema)
    print("✅ Valid YAML parsed successfully:")
    print(f"  User: {parsed['user']['name'].data}")
    print(f"  Email: {parsed['user']['email'].data}")
    print(f"  Website: {parsed['user']['website'].data}")
    print(f"  Phone: {parsed['user']['phone'].data if 'phone' in parsed['user'] else 'Not provided'}")
    print(f"  Languages: {parsed['settings']['languages'].data}")
    print(f"  Permissions: {[p.data for p in parsed['permissions']]}")
    
except YAMLValidationError as e:
    print(f"❌ Valid YAML failed: {e}")
    raise  # the valid document must parse

# Test invalid YAML documents
print(f"\nError Handling Tests:")
# What each document must be rejected for (strictyaml 1.7.3's own wording, measured)
expected_problems = {
    "invalid_email": "when expecting an email address",
    "invalid_url": "when expecting a URL",
    "invalid_type": "when expecting a boolean value",
}
for error_type, invalid_yaml in invalid_yamls.items():
    try:
        parsed = load(invalid_yaml, advanced_schema)
    except YAMLValidationError as e:
        assert expected_problems[error_type] in str(e), f"{error_type}: rejected for an unexpected reason: {e}"
        print(f"✅ {error_type}: Correctly caught error")
        print(f"   Reason: {str(e)[:80]}...")
        if hasattr(e, 'problem_mark'):
            print(f"   Line: {e.problem_mark.line + 1}")
    else:
        raise AssertionError(f"{error_type}: strictyaml accepted a document its schema forbids")

# Test optional fields behavior
optional_yaml = """
user:
  name: Eve
  email: eve@example.com
  website: https://eve.dev
settings:
  theme: purple
  notifications: false
  languages: rust,python
permissions:
  - read
"""

try:
    parsed = load(optional_yaml, advanced_schema)
    print(f"\n✅ Optional fields handling:")
    print(f"  Phone provided: {'phone' in parsed['user']}")
    print(f"  Bio provided: {'bio' in parsed['user']}")
    print(f"  Timeout provided: {'timeout' in parsed['settings']}")
    
except YAMLValidationError as e:
    print(f"❌ Optional fields test failed: {e}")
    raise  # a document that omits only Optional() keys must parse

print("✅ Advanced schema validation and error handling completed")

## Example 3: Configuration File Management and Updates

In [ ]:
# Example 3: Configuration File Management and Updates
from strictyaml import as_document
import tempfile
import os

# Configuration schema for an application
config_schema = Map({
    "app": Map({
        "name": Str(),
        "version": Str(),
        "debug": Bool(),
        Optional("port"): Int()
    }),
    "database": Map({
        "host": Str(),
        "port": Int(),
        "name": Str(),
        "ssl": Bool(),
        Optional("timeout"): Int()
    }),
    "logging": Map({
        "level": Str(),
        "format": Str(),
        "handlers": Seq(Str())
    })
})

# Initial configuration
initial_config = """
app:
  name: MyApp
  version: 1.0.0
  debug: false
  port: 8080
database:
  host: localhost
  port: 5432
  name: myapp_db
  ssl: true
logging:
  level: INFO
  format: "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
  handlers:
    - console
    - file
"""

print("Configuration File Management:")
print("=" * 40)

# Parse initial configuration
try:
    config = load(initial_config, config_schema)
    print("✅ Initial configuration loaded:")
    print(f"  App: {config['app']['name'].data} v{config['app']['version'].data}")
    print(f"  Debug mode: {config['app']['debug'].data}")
    print(f"  Database: {config['database']['host'].data}:{config['database']['port'].data}")
    print(f"  Logging level: {config['logging']['level'].data}")
    
    # Demonstrate programmatic updates
    print(f"\nConfiguration Updates:")
    
    # Update debug mode
    config['app']['debug'] = True
    print(f"✅ Debug mode updated: {config['app']['debug'].data}")
    
    # Update database timeout (add optional field)
    config['database']['timeout'] = 30
    print(f"✅ Database timeout added: {config['database']['timeout'].data}")
    
    # Update logging level
    config['logging']['level'] = "DEBUG"
    print(f"✅ Logging level updated: {config['logging']['level'].data}")
    
    # API drift: a strictyaml `YAML` sequence is a validated view, not a list — it has no
    # `.append`. Mutate through the parent key, which is how strictyaml keeps the document valid.
    config['logging']['handlers'] = [h.data for h in config['logging']['handlers']] + ["syslog"]
    handlers = [h.data for h in config['logging']['handlers']]
    print(f"✅ Logging handlers updated: {handlers}")
    
    # Generate updated YAML with preserved formatting
    updated_yaml = config.as_yaml()
    
    print(f"\nUpdated Configuration Preview:")
    print("─" * 30)
    print(updated_yaml[:300] + "..." if len(updated_yaml) > 300 else updated_yaml)
    
except Exception as e:
    print(f"❌ Configuration management failed: {e}")
    raise AssertionError(
        f"strictyaml example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

# Demonstrate file operations
print(f"\nFile Operations:")
with tempfile.NamedTemporaryFile(mode='w+', suffix='.yaml', delete=False) as temp_file:
    try:
        # Write configuration to file
        temp_file.write(initial_config)
        temp_file.flush()
        
        # Read and parse from file
        with open(temp_file.name, 'r') as f:
            file_content = f.read()
        
        file_config = load(file_content, config_schema)
        print(f"✅ Configuration read from file: {temp_file.name}")
        
        # Update configuration
        file_config['app']['version'] = "1.1.0"
        file_config['app']['debug'] = True
        
        # Write updated configuration back
        with open(temp_file.name, 'w') as f:
            f.write(file_config.as_yaml())
        
        print(f"✅ Configuration updated in file")
        
        # Verify updates
        with open(temp_file.name, 'r') as f:
            updated_content = f.read()
        
        verified_config = load(updated_content, config_schema)
        print(f"✅ Verification - Version: {verified_config['app']['version'].data}")
        print(f"✅ Verification - Debug: {verified_config['app']['debug'].data}")
        # the typed setters round-trip through as_yaml() and a re-validating load (measured: '1.1.0', True)
        assert verified_config['app']['version'].data == "1.1.0", verified_config['app']['version'].data
        assert verified_config['app']['debug'].data is True, verified_config['app']['debug'].data
        
    finally:
        # Clean up temporary file
        if os.path.exists(temp_file.name):
            os.unlink(temp_file.name)

# Configuration validation with environment overrides
print(f"\nEnvironment Override Simulation:")
env_overrides = {
    "APP_DEBUG": "true",
    "DATABASE_HOST": "production-db.example.com",
    "DATABASE_PORT": "5433",
    "LOGGING_LEVEL": "WARNING"
}

# Parse base configuration
base_config = load(initial_config, config_schema)

# Apply environment overrides
for env_key, env_value in env_overrides.items():
    if env_key == "APP_DEBUG":
        base_config['app']['debug'] = env_value.lower() == "true"
    elif env_key == "DATABASE_HOST":
        base_config['database']['host'] = env_value
    elif env_key == "DATABASE_PORT":
        base_config['database']['port'] = int(env_value)
    elif env_key == "LOGGING_LEVEL":
        base_config['logging']['level'] = env_value

print(f"✅ Environment overrides applied:")
print(f"  Debug: {base_config['app']['debug'].data}")
print(f"  DB Host: {base_config['database']['host'].data}")
print(f"  DB Port: {base_config['database']['port'].data}")
print(f"  Log Level: {base_config['logging']['level'].data}")
# each override is revalidated by the schema and read back typed (measured)
assert (base_config['app']['debug'].data, base_config['database']['host'].data, base_config['database']['port'].data,
        base_config['logging']['level'].data) == (True, "production-db.example.com", 5433, "WARNING")

print("✅ Configuration file management and updates completed")

## Example 4: Custom Validators and Complex Data Structures

In [ ]:
# Example 4: Custom Validators and Complex Data Structures
from strictyaml import Validator, Any, OrValidator, MapCombined, MapPattern, EmptyList, EmptyDict, Str, Map, Seq, Optional, Bool, load, YAMLValidationError
from datetime import datetime
import re

# Custom validator for date strings
class DateValidator(Validator):
    def validate_scalar(self, chunk):
        try:
            datetime.strptime(chunk.contents, "%Y-%m-%d")
            return chunk.contents
        except ValueError:
            chunk.expecting_but_found("date in YYYY-MM-DD format")
    
    def validate(self, chunk):
        """Required method that calls validate_scalar for scalar values"""
        chunk.expect_scalar("when expecting a date")
        return self.validate_scalar(chunk)

# Custom validator for semantic versions
class SemanticVersionValidator(Validator):
    def validate_scalar(self, chunk):
        pattern = r'^(\d+)\.(\d+)\.(\d+)(?:-([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?(?:\+([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?$'
        if re.match(pattern, chunk.contents):
            return chunk.contents
        else:
            chunk.expecting_but_found("semantic version (e.g., 1.2.3 or 1.0.0-alpha.1)")
    
    def validate(self, chunk):
        """Required method that calls validate_scalar for scalar values"""
        chunk.expect_scalar("when expecting a semantic version")
        return self.validate_scalar(chunk)

# Complex schema with custom validators
project_schema = Map({
    "project": Map({
        "name": Str(),
        "version": SemanticVersionValidator(),
        "description": Str(),
        "created": DateValidator(),
        "updated": DateValidator(),
        "tags": Seq(Str()) | EmptyList(),
        # open-ended mappings take MapPattern: Map({}) accepts only an EMPTY mapping, and a validator is not a Map key
        "metadata": MapPattern(Str(), Str()) | EmptyDict()
    }),
    "dependencies": OrValidator(
        MapPattern(Str(), SemanticVersionValidator()),
        EmptyDict()
    ),
    "scripts": OrValidator(
        MapPattern(Str(), Str()),
        EmptyDict()
    ),
    "environments": Seq(Map({
        "name": Str(),
        "type": Any(),  # Allow any type for environment type
        "config": MapPattern(Str(), Any()) | EmptyDict(),
        Optional("active"): Bool()
    })) | EmptyList()
})

# Complex YAML document
complex_yaml = """
project:
  name: awesome-project
  version: 2.1.3-beta.1
  description: An awesome project demonstrating StrictYAML
  created: 2023-01-15
  updated: 2023-09-15
  tags:
    - python
    - yaml
    - validation
  metadata:
    author: Development Team
    license: MIT
    repository: https://github.com/company/awesome-project

dependencies:
  requests: 2.31.0
  pyyaml: 6.0.1
  click: 8.1.7

scripts:
  test: python -m pytest tests/
  lint: flake8 src/
  build: python setup.py build
  deploy: python scripts/deploy.py

environments:
  - name: development
    type: local
    config:
      debug: true
      database_url: sqlite:///dev.db
    active: true
  - name: staging
    type: cloud
    config:
      debug: false
      database_url: postgresql://staging-db:5432/app
      replicas: 2
    active: false
  - name: production
    type: cloud
    config:
      debug: false
      database_url: postgresql://prod-db:5432/app
      replicas: 5
      monitoring: enabled
    active: true
"""

print("Custom Validators and Complex Data Structures:")
print("=" * 55)

try:
    parsed = load(complex_yaml, project_schema)
    
    print("✅ Complex YAML parsed successfully:")
    
    # Project information
    project = parsed['project']
    print(f"  Project: {project['name'].data} v{project['version'].data}")
    print(f"  Description: {project['description'].data}")
    print(f"  Created: {project['created'].data}")
    print(f"  Updated: {project['updated'].data}")
    print(f"  Tags: {[tag.data for tag in project['tags']]}")
    
    # Dependencies
    deps = parsed['dependencies']
    print(f"  Dependencies: {len(deps)} packages")
    for name, version in deps.items():
        print(f"    {name.data}: {version.data}")
    
    # Scripts
    scripts = parsed['scripts']
    print(f"  Scripts: {len(scripts)} commands")
    for name, command in scripts.items():
        print(f"    {name.data}: {command.data}")
    
    # Environments
    environments = parsed['environments']
    print(f"  Environments: {len(environments)} configured")
    for env in environments:
        active_status = "✅ Active" if env.get('active', False).data else "⏸️ Inactive"
        print(f"    {env['name'].data} ({env['type'].data}): {active_status}")
        
        # Show config keys
        config_keys = list(env['config'].keys()) if env['config'].data else []  # a YAML object has no truth value; test .data
        if config_keys:
            print(f"      Config: {[key.data for key in config_keys]}")

except YAMLValidationError as e:
    print(f"❌ Validation failed: {e}")
    raise  # the complex document is valid for this schema; a rejection is a regression

assert (project['version'].data, project['created'].data, len(deps), len(scripts), len(environments)) == ("2.1.3-beta.1", "2023-01-15", 3, 4, 3)

# Test custom validator edge cases
print(f"\nCustom Validator Testing:")

# Test date validator
test_dates = [
    ("2023-12-31", True, "Valid date"),
    ("2023-13-01", False, "Invalid month"),
    ("not-a-date", False, "Invalid format"),
    ("23-12-31", False, "Wrong format")
]

for date_str, should_pass, description in test_dates:
    test_yaml = f"date: {date_str}"
    test_schema = Map({"date": DateValidator()})
    
    try:
        result = load(test_yaml, test_schema)
    except YAMLValidationError:
        assert not should_pass, f"Date '{date_str}' ({description}) is valid but was rejected"
        print(f"  Date '{date_str}': ✅ Correctly failed - {description}")
    else:
        assert should_pass, f"Date '{date_str}' ({description}) is invalid but was accepted"
        print(f"  Date '{date_str}': ✅ Passed - {description}")

# Test version validator
test_versions = [
    ("1.0.0", True, "Basic semver"),
    ("2.1.3-beta.1", True, "Prerelease"),
    ("1.0.0+build.123", True, "Build metadata"),
    ("v1.0.0", False, "With 'v' prefix"),
    ("1.0", False, "Missing patch version")
]

for version_str, should_pass, description in test_versions:
    test_yaml = f"version: {version_str}"
    test_schema = Map({"version": SemanticVersionValidator()})
    
    try:
        result = load(test_yaml, test_schema)
    except YAMLValidationError:
        assert not should_pass, f"Version '{version_str}' ({description}) is valid but was rejected"
        print(f"  Version '{version_str}': ✅ Correctly failed - {description}")
    else:
        assert should_pass, f"Version '{version_str}' ({description}) is invalid but was accepted"
        print(f"  Version '{version_str}': ✅ Passed - {description}")

print("✅ Custom validators and complex data structures completed")

In [ ]:
# Example 5: Production Configuration Management System
from strictyaml import as_document, load, Map, Str, Int, Bool, Float, Seq, Optional, Validator, YAMLValidationError
import tempfile
import os
from pathlib import Path
import re

# Re-define SemanticVersionValidator for Example 5
class SemanticVersionValidator(Validator):
    def validate_scalar(self, chunk):
        pattern = r'^(\d+)\.(\d+)\.(\d+)(?:-([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?(?:\+([0-9A-Za-z-]+(?:\.[0-9A-Za-z-]+)*))?$'
        if re.match(pattern, chunk.contents):
            return chunk.contents
        else:
            chunk.expecting_but_found("semantic version (e.g., 1.2.3 or 1.0.0-alpha.1)")
    
    def validate(self, chunk):
        """Required method that calls validate_scalar for scalar values"""
        chunk.expect_scalar("when expecting a semantic version")
        return self.validate_scalar(chunk)
    
    def to_yaml(self, data):
        """Required method for serialization back to YAML"""
        return str(data)

# Production-grade configuration schema
production_schema = Map({
    "service": Map({
        "name": Str(),
        "version": SemanticVersionValidator(),
        "environment": Str(),
        "region": Str()
    }),
    "server": Map({
        "host": Str(),
        "port": Int(),
        "workers": Int(),
        "timeout": Int(),
        "ssl": Map({
            "enabled": Bool(),
            Optional("cert_path"): Str(),
            Optional("key_path"): Str()
        })
    }),
    "database": Map({
        "primary": Map({
            "host": Str(),
            "port": Int(),
            "name": Str(),
            "user": Str(),
            "ssl": Bool(),
            "pool_size": Int()
        }),
        Optional("replica"): Map({
            "host": Str(),
            "port": Int(),
            "name": Str(),
            "user": Str(),
            "ssl": Bool()
        })
    }),
    "cache": Map({
        "redis": Map({
            "host": Str(),
            "port": Int(),
            "db": Int(),
            "timeout": Int()
        })
    }),
    "monitoring": Map({
        "enabled": Bool(),
        "metrics_port": Int(),
        "log_level": Str(),
        "alerts": Seq(Map({
            "name": Str(),
            "threshold": Float(),
            "enabled": Bool()
        }))
    }),
    "features": Map({
        "rate_limiting": Bool(),
        "authentication": Bool(),
        "compression": Bool(),
        "maintenance_mode": Bool()
    })
})

# Production configuration template
production_config = """
service:
  name: user-service
  version: 3.2.1
  environment: production
  region: us-east-1

server:
  host: 0.0.0.0
  port: 8080
  workers: 4
  timeout: 30
  ssl:
    enabled: true
    cert_path: /etc/ssl/certs/service.crt
    key_path: /etc/ssl/private/service.key

database:
  primary:
    host: prod-db-primary.example.com
    port: 5432
    name: userdb
    user: app_user
    ssl: true
    pool_size: 20
  replica:
    host: prod-db-replica.example.com
    port: 5432
    name: userdb
    user: readonly_user
    ssl: true

cache:
  redis:
    host: redis-cluster.example.com
    port: 6379
    db: 0
    timeout: 5

monitoring:
  enabled: true
  metrics_port: 9090
  log_level: INFO
  alerts:
    - name: high_cpu_usage
      threshold: 80.0
      enabled: true
    - name: memory_leak
      threshold: 90.0
      enabled: true
    - name: response_time
      threshold: 2.0
      enabled: true

features:
  rate_limiting: true
  authentication: true
  compression: true
  maintenance_mode: false
"""

print("Production Configuration Management System:")
print("=" * 55)

# Create temporary configuration files for different environments
environments = {
    "development": {
        "server.workers": 1,
        "server.host": "127.0.0.1",
        "database.primary.host": "localhost",
        "database.primary.pool_size": 5,
        "monitoring.log_level": "DEBUG",
        "features.rate_limiting": False
    },
    "staging": {
        "server.workers": 2,
        "database.primary.host": "staging-db.example.com",
        "database.primary.pool_size": 10,
        "monitoring.log_level": "INFO",
        "features.maintenance_mode": False
    },
    "production": {
        "server.workers": 8,
        "database.primary.pool_size": 50,
        "monitoring.log_level": "WARNING",
        "features.maintenance_mode": False
    }
}

try:
    # Parse base configuration
    base_config = load(production_config, production_schema)
    print("✅ Base production configuration loaded")
    
    # Configuration management functions
    def apply_environment_overrides(config, env_name, overrides):
        """Apply environment-specific configuration overrides"""
        config_copy = load(config.as_yaml(), production_schema)
        
        for key_path, value in overrides.items():
            keys = key_path.split('.')
            current = config_copy
            
            # Navigate to the parent of the target key
            for key in keys[:-1]:
                current = current[key]
            
            # Set the final value
            final_key = keys[-1]
            current[final_key] = value
        
        return config_copy
    
    def validate_configuration(config):
        """Validate configuration for production readiness"""
        issues = []
        
        # Check SSL in production
        if config['service']['environment'].data == 'production':
            if not config['server']['ssl']['enabled'].data:
                issues.append("SSL must be enabled in production")
        
        # Check worker count
        workers = config['server']['workers'].data
        if workers < 1:
            issues.append("Worker count must be at least 1")
        
        # Check database pool size
        pool_size = config['database']['primary']['pool_size'].data
        if pool_size < workers:
            issues.append(f"Pool size ({pool_size}) should be >= workers ({workers})")
        
        # Check monitoring in production
        if (config['service']['environment'].data == 'production' and 
            not config['monitoring']['enabled'].data):
            issues.append("Monitoring must be enabled in production")
        
        return issues
    
    def get_config_summary(config):
        """Generate configuration summary"""
        return {
            'service': f"{config['service']['name'].data} v{config['service']['version'].data}",
            'environment': config['service']['environment'].data,
            'server': f"{config['server']['host'].data}:{config['server']['port'].data} ({config['server']['workers'].data} workers)",
            'database': f"{config['database']['primary']['host'].data}:{config['database']['primary']['port'].data}",
            'ssl_enabled': config['server']['ssl']['enabled'].data,
            'monitoring': config['monitoring']['enabled'].data,
            'features': {
                'auth': config['features']['authentication'].data,
                'rate_limit': config['features']['rate_limiting'].data,
                'maintenance': config['features']['maintenance_mode'].data
            }
        }
    
    # Test environment-specific configurations
    print(f"\nEnvironment Configuration Testing:")
    for env_name, overrides in environments.items():
        print(f"\n{env_name.upper()} Environment:")
        
        # Apply environment overrides
        env_config = apply_environment_overrides(base_config, env_name, overrides)
        
        # Update environment name
        env_config['service']['environment'] = env_name
        
        # Validate configuration
        validation_issues = validate_configuration(env_config)
        
        if validation_issues:
            print(f"  ❌ Configuration issues:")
            for issue in validation_issues:
                print(f"    - {issue}")
        else:
            print(f"  ✅ Configuration valid")
        
        # Show summary
        summary = get_config_summary(env_config)
        print(f"  Service: {summary['service']}")
        print(f"  Server: {summary['server']}")
        print(f"  SSL: {'Enabled' if summary['ssl_enabled'] else 'Disabled'}")
        print(f"  Monitoring: {'Enabled' if summary['monitoring'] else 'Disabled'}")
        print(f"  Features: Auth={summary['features']['auth']}, RateLimit={summary['features']['rate_limit']}")
    
    # Configuration file operations
    print(f"\nConfiguration File Operations:")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        config_dir = Path(temp_dir)
        
        # Save configurations for each environment
        for env_name, overrides in environments.items():
            env_config = apply_environment_overrides(base_config, env_name, overrides)
            env_config['service']['environment'] = env_name
            
            config_file = config_dir / f"config-{env_name}.yaml"
            
            with open(config_file, 'w') as f:
                f.write(env_config.as_yaml())
            
            print(f"  ✅ Saved {env_name} configuration: {config_file.name}")
            
            # Verify file can be read back
            with open(config_file, 'r') as f:
                file_content = f.read()
            
            reloaded_config = load(file_content, production_schema)
            assert reloaded_config['service']['environment'].data == env_name
        
        # Configuration comparison
        dev_file = config_dir / "config-development.yaml"
        prod_file = config_dir / "config-production.yaml"
        
        with open(dev_file, 'r') as f:
            dev_config = load(f.read(), production_schema)
        
        with open(prod_file, 'r') as f:
            prod_config = load(f.read(), production_schema)
        
        # Compare key differences
        print(f"\nDevelopment vs Production Differences:")
        print(f"  Workers: {dev_config['server']['workers'].data} vs {prod_config['server']['workers'].data}")
        print(f"  Pool Size: {dev_config['database']['primary']['pool_size'].data} vs {prod_config['database']['primary']['pool_size'].data}")
        print(f"  Log Level: {dev_config['monitoring']['log_level'].data} vs {prod_config['monitoring']['log_level'].data}")
        print(f"  Rate Limiting: {dev_config['features']['rate_limiting'].data} vs {prod_config['features']['rate_limiting'].data}")
    
    # Configuration template generation
    print(f"\nConfiguration Template Features:")
    
    # Generate minimal configuration template
    minimal_config = as_document({
        'service': {
            'name': 'new-service',
            'version': '1.0.0',
            'environment': 'development',
            'region': 'us-west-2'
        },
        'server': {
            'host': '127.0.0.1',
            'port': 8000,
            'workers': 1,
            'timeout': 30,
            'ssl': {'enabled': False}
        }
    })
    
    print(f"  ✅ Minimal template generated")
    print(f"  Template preview:")
    print("    " + "\n    ".join(minimal_config.as_yaml().split('\n')[:8]))
    
    # Configuration merging simulation
    override_yaml = """
server:
  port: 9000
  workers: 3
features:
  rate_limiting: true
"""
    
    base_dict = base_config.data
    override_config = load(override_yaml, Map({
        'server': Map({
            'port': Int(),
            'workers': Int()
        }),
        'features': Map({
            'rate_limiting': Bool()
        })
    }))
    
    print(f"  ✅ Configuration override simulation complete")
    print(f"  Original port: {base_config['server']['port'].data}")
    print(f"  Override port: {override_config['server']['port'].data}")

except Exception as e:
    print(f"❌ Production configuration management failed: {e}")
    raise AssertionError(
        f"strictyaml example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Production configuration management system completed")